# Notebook 05 - Train T5 for ASTE on 14res + 15res + 16res

Task: **Aspect Sentiment Triplet Extraction (ASTE)**.

Model input:

```text
The price is reasonable although the service is poor .
```

Model output:

```text
aspect: price | opinion: reasonable | sentiment: positive ; aspect: service | opinion: poor | sentiment: negative
```

Dataset format:

```text
sentence #### aspect tags #### opinion tags
```

`dev.txt` la **validation set**: dung de chon best checkpoint trong luc train, khong dung de train truc tiep va khong dung lam test cuoi.

In [ ]:
import importlib.util, subprocess, sys

required = ["transformers", "datasets", "accelerate", "sklearn", "pandas", "matplotlib", "seaborn", "sentencepiece"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
import os
import re
import ast
import json
import random
import inspect
import shutil
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.utils import logging as hf_logging

warnings.filterwarnings("ignore", category=FutureWarning)
hf_logging.set_verbosity_error()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 1. Config

In [ ]:
# t5-small is safe on Kaggle GPU. Same hyperparameters are reused as-is for the
# t5-base / flan-t5-base siblings in this comparison (notebooks/train-{t5-base,flan-t5-base}-for-aste-on-14res-15res-16res.ipynb) — only the checkpoint changes.
MODEL_NAME = "t5-small"
SHORT_NAME = "t5-small"        # used for output dir naming
MAX_INPUT_LENGTH = 160
MAX_TARGET_LENGTH = 160
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
EPOCHS = 20
LEARNING_RATE = 3e-4

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORKING_ROOT / f"{SHORT_NAME}-aste-restaurant"
BEST_MODEL_DIR = WORKING_ROOT / f"{SHORT_NAME}-aste-restaurant-best"
CLEAN_OUTPUT = True

if CLEAN_OUTPUT:
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    shutil.rmtree(BEST_MODEL_DIR, ignore_errors=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 2. Locate 14res / 15res / 16res Files

Notebook se uu tien dataset da add trong `/kaggle/input`. Neu khong thay, no se clone repo ASTE tu GitHub.

In [ ]:
DOMAINS = ["14res", "15res", "16res"]
KAGGLE_DOMAIN_DIRS = {
    "14res": INPUT_ROOT / "semi-triple-14res",
    "15res": INPUT_ROOT / "semi-triple-15res",
    "16res": INPUT_ROOT / "semi-triple-16res",
}

def find_domain_dir(root: Path, domain: str):
    if not root.exists():
        return None
    candidates = []

    explicit_dir = KAGGLE_DOMAIN_DIRS.get(domain)
    if explicit_dir is not None and explicit_dir.exists() and any(explicit_dir.glob("*.txt")):
        candidates.append(explicit_dir)

    for p in root.rglob("*"):
        if p.is_dir() and domain in p.name.lower() and any(p.glob("*.txt")):
            candidates.append(p)
    return sorted(candidates)[0] if candidates else None

def find_split_file(domain_dir: Path, split: str):
    files = sorted(domain_dir.glob("*.txt"))
    names = [f.name.lower() for f in files]

    if split == "train":
        preferred = ["train.txt", f"{domain_dir.name}_train.txt", f"{domain_dir.name}t_train.txt", f"{domain_dir.name}rest_train.txt"]
        patterns = ["train"]
    elif split == "dev":
        preferred = ["dev.txt", "val.txt", f"{domain_dir.name}_dev.txt", f"{domain_dir.name}t_dev.txt", f"{domain_dir.name}rest_dev.txt"]
        patterns = ["dev", "val"]
    elif split == "test":
        preferred = ["test.txt", f"{domain_dir.name}_test.txt", f"{domain_dir.name}t_test.txt", f"{domain_dir.name}rest_test.txt"]
        patterns = ["test"]
    else:
        raise ValueError(split)

    for name in preferred:
        for f in files:
            if f.name.lower() == name.lower():
                return f

    for f, name in zip(files, names):
        if any(pat in name for pat in patterns):
            return f
    return None

def collect_dataset_files(root: Path):
    found = {}
    for domain in DOMAINS:
        d = find_domain_dir(root, domain)
        if d is None:
            continue
        split_files = {split: find_split_file(d, split) for split in ["train", "dev", "test"]}
        if all(split_files.values()):
            found[domain] = split_files
    return found

dataset_files = collect_dataset_files(INPUT_ROOT)

if len(dataset_files) < 3:
    repo_dir = WORKING_ROOT / "SemEval-Triplet-data"
    if not repo_dir.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/xuuuluuu/SemEval-Triplet-data.git",
            str(repo_dir),
        ])
    dataset_files = collect_dataset_files(repo_dir)

print(json.dumps({d: {s: str(p) for s, p in splits.items()} for d, splits in dataset_files.items()}, indent=2))
missing_domains = [d for d in DOMAINS if d not in dataset_files]
assert not missing_domains, f"Missing domains: {missing_domains}. Add dataset to Kaggle Input or enable Internet."

## 3. Parse ASTE Tag Format

In [ ]:
SENTIMENT_MAP = {"POS": "positive", "NEG": "negative", "NEU": "neutral"}

def split_token_tag(item: str):
    token, tag = item.rsplit("=", 1)
    return token, tag

def parse_tag_sequence(tag_text: str):
    return [split_token_tag(item) for item in tag_text.strip().split()]

def phrase_from_tokens(tokens):
    return " ".join(tokens).replace(" n't", "n't").replace(" 's", "'s").strip()

def parse_aste_line(line: str):
    parts = line.strip().split("####")
    if len(parts) != 3:
        return None

    sentence, target_tag_text, opinion_tag_text = parts
    target_pairs = parse_tag_sequence(target_tag_text)
    opinion_pairs = parse_tag_sequence(opinion_tag_text)

    target_groups = {}
    for token, tag in target_pairs:
        if tag == "O":
            continue
        if "-" not in tag:
            continue
        group_id, sentiment_code = tag.split("-", 1)
        target_groups.setdefault(group_id, {"tokens": [], "sentiment": sentiment_code})
        target_groups[group_id]["tokens"].append(token)

    opinion_groups = {}
    for token, tag in opinion_pairs:
        if tag == "O":
            continue
        opinion_groups.setdefault(tag, [])
        opinion_groups[tag].append(token)

    triplets = []
    for group_id, target_info in sorted(target_groups.items(), key=lambda x: (len(x[0]), x[0])):
        opinion_group_id = "S" * len(group_id)
        aspect = phrase_from_tokens(target_info["tokens"])
        opinion = phrase_from_tokens(opinion_groups.get(opinion_group_id, []))
        sentiment = SENTIMENT_MAP.get(target_info["sentiment"], target_info["sentiment"].lower())
        if aspect and opinion:
            triplets.append({"aspect": aspect, "opinion": opinion, "sentiment": sentiment})
    return {"sentence": sentence.strip(), "triplets": triplets}

def triplets_to_text(triplets):
    if not triplets:
        return "no triplet"
    chunks = []
    for t in triplets:
        chunks.append(f"aspect: {t['aspect']} | opinion: {t['opinion']} | sentiment: {t['sentiment']}")
    return " ; ".join(chunks)

sample_path = dataset_files["14res"]["train"]
with open(sample_path, "r", encoding="utf-8") as f:
    sample = parse_aste_line(f.readline())
print(sample)
print(triplets_to_text(sample["triplets"]))

## 4. Build Train / Dev / Test DataFrames

Ta merge train cua 14res, 15res, 16res thanh mot train set. Tuong tu voi dev va test.

In [ ]:
def load_split(domain: str, split: str, path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            item = parse_aste_line(line)
            if item is None:
                continue
            rows.append({
                "domain": domain,
                "split": split,
                "row_id": f"{domain}-{split}-{idx}",
                "text": item["sentence"],
                "target_text": triplets_to_text(item["triplets"]),
                "triplets": item["triplets"],
                "triplet_count": len(item["triplets"]),
            })
    return pd.DataFrame(rows)

frames = []
for domain, splits in dataset_files.items():
    for split, path in splits.items():
        frames.append(load_split(domain, split, path))

all_df = pd.concat(frames, ignore_index=True)
train_df = all_df[all_df["split"] == "train"].reset_index(drop=True)
dev_df = all_df[all_df["split"] == "dev"].reset_index(drop=True)
test_df = all_df[all_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Dev  :", dev_df.shape)
print("Test :", test_df.shape)
print("\nRows by domain/split:")
display(all_df.groupby(["domain", "split"]).size().unstack(fill_value=0))
print("\nTriplet count distribution:")
display(train_df["triplet_count"].value_counts().sort_index())
display(train_df[["text", "target_text"]].head(10))

## 5. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

PREFIX = "extract aspect sentiment triplets: "

def to_hf_dataset(df):
    return Dataset.from_pandas(df[["text", "target_text"]].reset_index(drop=True))

def preprocess_batch(batch):
    inputs = [PREFIX + text for text in batch["text"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch["target_text"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = to_hf_dataset(train_df).map(preprocess_batch, batched=True, remove_columns=["text", "target_text"])
dev_ds = to_hf_dataset(dev_df).map(preprocess_batch, batched=True, remove_columns=["text", "target_text"])
test_ds = to_hf_dataset(test_df).map(preprocess_batch, batched=True, remove_columns=["text", "target_text"])

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

## 6. Metrics

In [ ]:
TRIPLET_RE = re.compile(r"aspect:\s*(.*?)\s*\|\s*opinion:\s*(.*?)\s*\|\s*sentiment:\s*(positive|negative|neutral)", re.IGNORECASE)

def normalize_text(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())

def parse_triplet_text(text):
    triples = set()
    if normalize_text(text) == "no triplet":
        return triples
    for match in TRIPLET_RE.finditer(text):
        aspect, opinion, sentiment = match.groups()
        triples.add((normalize_text(aspect), normalize_text(opinion), normalize_text(sentiment)))
    return triples

def triplet_prf(pred_texts, gold_texts):
    tp = pred_total = gold_total = 0
    for pred, gold in zip(pred_texts, gold_texts):
        pred_set = parse_triplet_text(pred)
        gold_set = parse_triplet_text(gold)
        tp += len(pred_set & gold_set)
        pred_total += len(pred_set)
        gold_total += len(gold_set)
    precision = tp / pred_total if pred_total else 0.0
    recall = tp / gold_total if gold_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

def safe_token_ids(sequences):
    arr = np.asarray(sequences)
    if arr.ndim == 3:
        arr = np.argmax(arr, axis=-1)
    arr = np.where(arr == -100, tokenizer.pad_token_id, arr)
    arr = np.where(arr < 0, tokenizer.pad_token_id, arr)
    arr = np.where(arr >= len(tokenizer), tokenizer.pad_token_id, arr)
    return arr.astype(np.int64)

def safe_decode_batch(sequences):
    return tokenizer.batch_decode(safe_token_ids(sequences), skip_special_tokens=True)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    pred_texts = safe_decode_batch(preds)
    gold_texts = safe_decode_batch(labels)
    exact_match = np.mean([normalize_text(p) == normalize_text(g) for p, g in zip(pred_texts, gold_texts)])
    precision, recall, f1 = triplet_prf(pred_texts, gold_texts)
    return {
        "exact_match": float(exact_match),
        "triplet_precision": precision,
        "triplet_recall": recall,
        "triplet_f1": f1,
    }

## 7. Train

In [ ]:
args_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=1,
    logging_strategy="steps",
    logging_steps=100,
    disable_tqdm=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    load_best_model_at_end=True,
    metric_for_best_model="triplet_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

if "eval_strategy" in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters:
    args_kwargs["eval_strategy"] = "epoch"
else:
    args_kwargs["evaluation_strategy"] = "epoch"

if "save_only_model" in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters:
    args_kwargs["save_only_model"] = True

training_args = Seq2SeqTrainingArguments(**args_kwargs)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 8. Evaluate on Dev and Test

In [ ]:
dev_metrics = trainer.evaluate(dev_ds)
test_metrics = trainer.evaluate(test_ds)
print("Dev metrics:")
print(dev_metrics)
print("\nTest metrics:")
print(test_metrics)

In [ ]:
pred_output = trainer.predict(test_ds)
pred_texts = safe_decode_batch(pred_output.predictions)
gold_texts = safe_decode_batch(pred_output.label_ids)

pred_df = test_df.copy()
pred_df["prediction"] = pred_texts
pred_df["gold"] = gold_texts
pred_df["exact"] = [normalize_text(p) == normalize_text(g) for p, g in zip(pred_texts, gold_texts)]

display(pred_df[["domain", "text", "gold", "prediction", "exact"]].head(30))
print("Exact match:", pred_df["exact"].mean())

## 9. Save Best Model

In [ ]:
# load_best_model_at_end=True, so trainer.model is the best checkpoint according to dev triplet_f1.
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

metrics_df = pd.DataFrame([
    {"split": "dev", **dev_metrics},
    {"split": "test", **test_metrics},
])
metrics_df.to_csv(BEST_MODEL_DIR / "metrics.csv", index=False)
pred_df.to_csv(BEST_MODEL_DIR / "test_predictions.csv", index=False)

print("Saved best model and outputs to:", BEST_MODEL_DIR)
display(metrics_df)

## 10. Try New Sentences

In [ ]:
def predict_aste(sentence, num_beams=4):
    inputs = tokenizer(PREFIX + sentence, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH).to(model.device)
    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=MAX_TARGET_LENGTH,
            num_beams=num_beams,
            early_stopping=True,
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return text, parse_triplet_text(text)

examples = [
    "The price is reasonable although the service is poor .",
    "The food was delicious but the waiter was rude .",
    "Great atmosphere , friendly staff , and terrible waiting time .",
]

for ex in examples:
    pred, triples = predict_aste(ex)
    print("Sentence:", ex)
    print("Prediction:", pred)
    print("Parsed:", triples)
    print()